# P1 — ¿Qué es el over-squashing?

Una GNN actualiza cada nodo mezclando a sus vecinos, un salto a la vez. Cuando un nodo necesita información de
*lejos* y el grafo tiene un **cuello de botella**, una cantidad enorme de información distante se aprieta en un
vector de tamaño fijo. Ese apretujamiento es el **over-squashing**.

**Figuras:** (1) el grafo cuello de botella, (2) cómo explotan los mensajes con la profundidad, (3) el mapa de
calor de multiplicidad.

In [ ]:
import torch, numpy as np
from oversquash import viz
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.ideal_bridge import walk_operators
K, M = 4, 3

## Figura 1 — el cuello de botella

Las fuentes (azul) pasan por un medio estrecho (amarillo) hacia un único objetivo (verde). Todo camino cruza
la parte estrecha.

In [ ]:
data = make_bottleneck_graph(K, M, 2, torch.Generator().manual_seed(0))
viz.draw_bottleneck(data, K, M, title='fuentes -> cuello de botella -> objetivo')

## Figura 2 — los mensajes explotan con la profundidad

Cuenta los recorridos de longitud `profundidad+1` que llegan al objetivo. Una GNN normal debe comprimir
*todos* en el único vector del objetivo. Mira cómo el conteo se dispara al hacer el grafo más profundo (escala
logarítmica).

In [ ]:
depths = [1,2,3,4]; msgs = []
for d in depths:
    dd = make_bottleneck_graph(K, M, d, torch.Generator().manual_seed(0))
    raw, _ = walk_operators(dd.edge_index.numpy(), dd.num_nodes, max_length=d+1)
    t = int(dd.root_mask.nonzero()[0]); msgs.append(int(raw[d][:, t].sum()))
viz.plot_message_explosion(depths, msgs, 'mensajes apretados en un vector', 'profundidad', 'mensajes (log)')

## Figura 3 — de dónde vienen los mensajes

La matriz de multiplicidad en el rango más profundo: fila = objetivo, columna = fuente, color = número de
recorridos. La columna brillante hacia el objetivo es la acumulación que se aprieta.

In [ ]:
deep = make_bottleneck_graph(K, M, 3, torch.Generator().manual_seed(0))
raw, _ = walk_operators(deep.edge_index.numpy(), deep.num_nodes, max_length=4)
viz.plot_multiplicity_heatmap(raw[3], 'multiplicidad de recorridos en rango 4 (A^4)')
print('El conteo de caminos crece ~K*M^d, pero el vector del objetivo no cambia de tamaño. Sigamos con P2.')